# PyTorch vs PyTorch Lightning: CIFAR-10 Classification Comparison
## This notebook demonstrates the same Object recognition task (CIFAR-10) implemented in both frameworks

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# CIFAR-10 classes
CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')



# PART 1: PURE PYTORCH IMPLEMENTATION
## 1.1 Define CNN Model
Define a simple CNN with 3 Convolutional layers + Batch Normalization + ReLU + MaxPool

In [ ]:

class PyTorchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Pooling and normalization
        self.pool = nn.MaxPool2d(2, 2)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Flatten and FC layers
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## 1.2 Prepare Data with Augmentation


Prepare CIFAR-10 dataset class.

CIFAR is a collection 60,000 32x32 pixel color images used for training computer vision tasks like image classification over 10 classes.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Download and load datasets
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

# Split train into train and validation
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=128, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=128, num_workers=4)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 1.3 Training Loop (Manual)


Implement the training loop with Pytorch.
Be careful to explicitly:
- Manage devices
- Support logging
- Set parameters
- Compute loss
- Compute gradient
- Perform backpropagation
- Set model in train/eval mode for train/val steps
- Update learning rate
- Save best model

In [ ]:
def train_pytorch(model, train_loader, val_loader, epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                           factor=0.5, patience=2)

    best_val_acc = 0.0

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = output.max(1)
            train_total += target.size(0)
            train_correct += predicted.eq(target).sum().item()

            # Print progress every 50 batches
            if batch_idx % 50 == 0:
                print(f'  Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        # Update learning rate
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f'\nEpoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print(f'  Learning Rate: {current_lr:.6f}')

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_pytorch_model.pth')
            print(f'  ✓ New best model saved! (Val Acc: {val_acc:.2f}%)')
        print()

    return model

## 1.4 Train the Pytorch Model

In [ ]:
pytorch_model = PyTorchCNN()
print("\nTraining PyTorch model...")
pytorch_model = train_pytorch(pytorch_model, train_loader, val_loader, epochs=10)

## 1.5 Testing loop


Implement the testing loop with Pytorch.
Be careful to explicitly:
- Manage devices
- Support logging
- Set parameters
- Compute metrics
- Set model in eval mode

In [ ]:
def test_pytorch(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0
    criterion = nn.CrossEntropyLoss()

    # Per-class accuracy tracking
    class_correct = [0] * 10
    class_total = [0] * 10

    print("\nTesting PyTorch model...")
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            test_loss += loss.item()
            _, predicted = output.max(1)
            test_total += target.size(0)
            test_correct += predicted.eq(target).sum().item()

            # Per-class accuracy
            for i in range(len(target)):
                label = target[i]
                class_correct[label] += (predicted[i] == label).item()
                class_total[label] += 1

    test_acc = 100. * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f'\n{"="*60}')
    print(f'PyTorch Test Results:')
    print(f'{"="*60}')
    print(f'Test Loss: {avg_test_loss:.4f}')
    print(f'Test Accuracy: {test_acc:.2f}% ({test_correct}/{test_total})')
    print(f'\nPer-class Accuracy:')
    for i in range(10):
        acc = 100. * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        print(f'  {CLASSES[i]:>8s}: {acc:5.2f}%')
    print(f'{"="*60}\n')



## 1.6 Test the Pytorch model

In [ ]:
test_pytorch(pytorch_model, test_loader)

# PART 2: PYTORCH LIGHTNING IMPLEMENTATION
## 2.1 Define the Lightning CNN Module

In [ ]:
! pip install lightning

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Implement the Lightning Module for the same CNN.

N.B.: Write how to configure optimizers and how the model should work in train, validation or test steps

In [ ]:
class LightningCNN(pl.LightningModule):
    def __init__(self, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Pooling and normalization
        self.pool = nn.MaxPool2d(2, 2)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Flatten and FC layers
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        # Log metrics
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        # Log metrics
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        # Log metrics
        self.log('test_loss', loss)
        self.log('test_acc', acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }


### ALTERNATIVE passing the already-defined model as argument in the class constructor

# class LightningCNN(pl.LightningModule):
#     def __init__(self, model, learning_rate=0.001):
#         super().__init__()
#         self.model = model
#         self.save_hyperparameters()


#     def forward(self, x):
#         return self.model(x)

#     def training_step(self, batch, batch_idx):
#         x, y = batch
#         logits = self(x)
#         loss = F.cross_entropy(logits, y)

#         # Calculate accuracy
#         preds = torch.argmax(logits, dim=1)
#         acc = (preds == y).float().mean()

#         # Log metrics
#         self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
#         self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

#         return loss

#     def validation_step(self, batch, batch_idx):
#         x, y = batch
#         logits = self(x)
#         loss = F.cross_entropy(logits, y)

#         # Calculate accuracy
#         preds = torch.argmax(logits, dim=1)
#         acc = (preds == y).float().mean()

#         # Log metrics
#         self.log('val_loss', loss, prog_bar=True)
#         self.log('val_acc', acc, prog_bar=True)

#     def test_step(self, batch, batch_idx):
#         x, y = batch
#         logits = self(x)
#         loss = F.cross_entropy(logits, y)

#         # Calculate accuracy
#         preds = torch.argmax(logits, dim=1)
#         acc = (preds == y).float().mean()

#         # Log metrics
#         self.log('test_loss', loss)
#         self.log('test_acc', acc)

#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
#         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.5, patience=2
#         )
#         return {
#             'optimizer': optimizer,
#             'lr_scheduler': {
#                 'scheduler': scheduler,
#                 'monitor': 'val_loss'
#             }
#         }

## 2.2 Create DataModule


Implement the DataModule for CIFAR-10.

N.B. Do it in order to reuse the same code if you'll work in the future with the same dataset!

In [ ]:
class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(self, batch_size=128):
        super().__init__()
        self.batch_size = batch_size

        # Data augmentation for training
        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

        # No augmentation for validation/test
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

    def prepare_data(self):
        # Download data (called only once on a single GPU)
        datasets.CIFAR10('./data', train=True, download=True)
        datasets.CIFAR10('./data', train=False, download=True)

    def setup(self, stage=None):
        # Load and split data (called on every GPU)
        if stage == 'fit' or stage is None:
            cifar_full = datasets.CIFAR10('./data', train=True, transform=self.train_transform)
            train_size = int(0.9 * len(cifar_full))
            val_size = len(cifar_full) - train_size
            self.train_dataset, self.val_dataset = random_split(
                cifar_full, [train_size, val_size]
            )

        if stage == 'test' or stage is None:
            self.test_dataset = datasets.CIFAR10('./data', train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                         shuffle=True, num_workers=4, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                         num_workers=4, persistent_workers=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                         num_workers=4, persistent_workers=True)



## 2.3 Define Callbacks and Trainer


Define how the model should be trained. You can define callbacks (for logging ,save best model checkpoints, early stopping, etc.)

In [ ]:
lightning_model = LightningCNN(learning_rate=0.001)
data_module = CIFAR10DataModule(batch_size=128)

# Configure callbacks
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    dirpath='checkpoints/',
    filename='cifar10-{epoch:02d}-{val_acc:.2f}',
    save_top_k=1,
    mode='max'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    verbose=True
)

# Create trainer
trainer = pl.Trainer(
    max_epochs=10,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator='auto', # 'cpu', 'cuda'
    devices=1,
    log_every_n_steps=50,
    precision='16-mixed' if torch.cuda.is_available() else 32  # Use mixed precision on GPU
)

##2.4 Train with Lightning


In [ ]:
print("\nTraining Lightning model...")
trainer.fit(lightning_model, data_module)

# Test the model
print("\nTesting Lightning model...")
trainer.test(lightning_model, data_module)

# Key Differences Summary

| PyTorch | PyTorch Lightning|
| ------------------------------| ---------------------------|
|✗ ~150+ lines of boilerplate code|✓ ~80 lines for model + data module|
|✗ Manual device management (CPU/GPU) | ✓ Automatic device management |
|✗ Manual training/validation loops| ✓ Automatic training/validation loops |
| ✗ Manual progress tracking and printing |✓ Built-in progress bars |
|✗ Manual checkpoint saving logic | ✓ Built-in checkpointing (ModelCheckpoint) |
|✗ Manual learning rate scheduling | ✓ Easy scheduler integration |
| ✗ Manual early stopping implementation| ✓ Built-in early stopping callback |
| ✗ No built-in logging (TensorBoard, etc.) | ✓ Automatic logging to TensorBoard/MLflow/etc. |
|✗ Need to handle distributed training manually | ✓ Easy multi-GPU training (just change devices=N) |
|✓ Full control over every detail | ✓ Mixed precision training with one flag|
| |✓ Better code organization and readability |
| |✓ More reproducible experiments|


Estimated Code Reduction: ~50% less boilerplate

Training Speed: Same (Lightning adds minimal overhead)

GPU Utilization: Better with Lightning (mixed precision, optimizations)


**Key Benefits for CIFAR-10**

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

- Data augmentation handled cleanly in DataModule
- Learning rate scheduling integrated seamlessly
- Easy to add callbacks without touching training code
- Automatic gradient accumulation and clipping
- Built-in validation during training
- Professional-grade experiment tracking
- Scales effortlessly to larger datasets/models





In [ ]:
## EXTRA
# load pytorch weights into Lightning model
lightning_model.load_state_dict(pytorch_model.state_dict())